<a href="https://colab.research.google.com/github/RuiRafael11/LLM-Engineering-Course/blob/main/week2/exercise2-synthetic-data/DatasetCreationWithOllama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate gradio

In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, TextIteratorStreamer
import torch
import gradio as gr
import threading

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [4]:
# Define the instruct model names
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [6]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [17]:
def chat (message, history):

  SYSTEM_PROMPT = """You are an expert synthetic data generator.
Your sole task is to generate high-quality datasets based on the user's request.
You must output the data strictly in a valid JSON format (a list of objects).
Do NOT include any introduction, explanations, markdown formatting (like ```json), or conversational text.
Return ONLY the raw JSON array.

Example format:
[
  {"instruction": "User prompt example", "response": "Expected model response"},
  {"instruction": "Another prompt example", "response": "Another expected response"}
]"""

  messages = [{"role": "system", "content" : SYSTEM_PROMPT }]
  for user, assistant in history:
    messages.append({"role": "user", "content": user})
    messages.append({"role": "assistant", "content": assistant})
  messages.append({"role": "user", "content": message})

  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
  streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

  generate_kwargs = dict(
      input_ids= inputs["input_ids"],
      streamer=streamer,
      max_new_tokens=1024,
      do_sample=True,
      temperature=0.85 )

  t = threading.Thread(target=model.generate, kwargs=generate_kwargs)
  t.start()

  full_text = ""
  for new_text in streamer:
    full_text += new_text
    yield full_text

In [18]:
demo = gr.ChatInterface (
    fn = chat,
    type= "messages",
    title="🤖 Synthetic Data Generator Chat",
    description = "Ask the model to generate datasets. Each response will be formatted strictly as raw JSON.",
    examples=[
        "Generate 3 examples of toxic vs non-toxic tweets for classification training.",
        "Generate 5 high-quality Python coding challenges with instructions and solutions.",
        "Generate 4 customer reviews for a broken smartphone, including a sentiment score (positive/negative)."
    ],
)
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a28def1b42c66dfa9d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a28def1b42c66dfa9d.gradio.live


In [19]:
with gr.Blocks() as demo:
    gr.Markdown("# 🤖 LLM Synthetic Data Generator")
    gr.Markdown("Enter the instructions for the dataset you want to generate below.")

    with gr.Row():
        with gr.Column():
            input_txt = gr.Textbox(
                label="Generation Prompt / Instructions",
                placeholder="Ex: Generate 5 examples of diverse customer support interactions regarding delayed shipping.",
                lines=3
            )
            btn_gerar = gr.Button("Generate Synthetic Data", variant="primary")

        with gr.Column():
            output_json = gr.Textbox(label="Generated Dataset (JSON)", lines=15)

    # Configuração do clique do botão:
    # 1. Mudamos a função para 'chat' (a sua função atualizada)
    # 2. Mantemos o gr.State([]) apenas para suprir o argumento 'history' que a função 'chat' espera receber
    btn_gerar.click(
        fn=chat,
        inputs=[input_txt, gr.State([])],
        outputs=output_json
    )

# Inicializa o painel no Colab com o modo debug ativo
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://39a56f7a999653ab9c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://39a56f7a999653ab9c.gradio.live


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
